# Query & Retrieval (Pipeline 004)

## Overview

Interactive query notebook for the RAG Job Failure pipeline. Retrieve relevant job failures and get contextual answers.

### Pipeline Order
`001` (setup) → `002` (ingestion) → `003` (embedding) → **`004`** (query)

## What This Notebook Does

- **Loads Embeddings:** Retrieves embeddings from `rag_jobs.job_embeddings` Delta table (written by notebook `003`)

- **Embeds Query:** Converts user query to vector using `databricks-gte-large-en` (same model used at index time)
  - Ensures semantic consistency between indexing and retrieval

- **Retrieves Results:** Finds top-k most relevant failure records using cosine similarity
  - Configurable via `top_k` widget (default: 5)
  - Returns matching error records with metadata and actions

- **Generates Answers:** Uses Databricks Foundation Model API LLM to create contextual responses
  - Multiple LLM options available (Llama 3.1, DBRX, Mixtral)
  - Generates insights based on retrieved failures and recommended actions

## Requirements

- **No pip install needed** — uses Databricks SDK and pre-installed libraries (DBR 14.3+)
- **No external API keys** — all models via Databricks Foundation Model API
- Pre-requisite: Notebooks `001`, `002`, and `003` must be run first


In [ ]:
# COMMAND ----------

# DBTITLE 1,Config & setup
# DBTITLE 1,Config & setup
# No pip install needed -- databricks-sdk and numpy are pre-installed in DBR 14.3+
from databricks.sdk import WorkspaceClient
import numpy as np

w = WorkspaceClient()

# --- Widgets ---
# llm_model : any Databricks Foundation Model API chat endpoint
# top_k     : number of failure records to retrieve per query
dbutils.widgets.dropdown(
    "llm_model",
    "databricks-meta-llama-3-1-70b-instruct",
    [
        "databricks-meta-llama-3-1-70b-instruct",
        "databricks-meta-llama-3-1-8b-instruct",
        "databricks-dbrx-instruct",
        "databricks-mixtral-8x7b-instruct",
    ],
    "LLM model"
)
dbutils.widgets.text("top_k", "5", "Top-k results")

EMBEDDINGS_TABLE = "rag_jobs.job_embeddings"  # Delta table written by notebook 003
EMBED_MODEL      = "databricks-gte-large-en"   # must match the model used in notebook 003
LLM_MODEL        = dbutils.widgets.get("llm_model")
TOP_K            = int(dbutils.widgets.get("top_k"))

print(f"Embeddings table : {EMBEDDINGS_TABLE}")
print(f"Embed model      : {EMBED_MODEL}")
print(f"LLM model        : {LLM_MODEL}")
print(f"Top-k            : {TOP_K}")





# COMMAND ----------

# DBTITLE 1,ask() and ask_with_sources() helpers
# DBTITLE 1,ask() and ask_with_sources() helpers
def ask(question: str, top_k: int = TOP_K) -> str:
    """Ask a question about job failures. Returns the LLM answer."""
    print(f"\n{'='*60}")
    print(f"Q: {question}")
    print(f"{'='*60}")
    docs    = retrieve(question, top_k)
    context = build_context(docs)
    answer  = call_llm(question, context)
    print(f"A: {answer}")
    return answer

def ask_with_sources(question: str, top_k: int = TOP_K) -> tuple[str, list[dict]]:
    """Ask a question and also print the retrieved source records."""
    print(f"\n{'='*60}")
    print(f"Q: {question}")
    print(f"{'='*60}")
    docs = retrieve(question, top_k)

    print(f"\nRetrieved {len(docs)} source records:")
    for i, doc in enumerate(docs):
        print(f"  [{i+1}] Job: {doc.get('job_name')} | "
              f"Date: {doc.get('run_date')} | "
              f"Error: {doc.get('error_type')} | "
              f"Score: {doc.get('score', 0):.3f}")

    context = build_context(docs)
    answer  = call_llm(question, context)
    print(f"\nAnswer:\n{answer}")
    return answer, docs

print("Query functions ready.")


In [ ]:
# COMMAND ----------

# DBTITLE 1,Retrieval helpers
# DBTITLE 1,Retrieval helpers
def embed(texts: list[str]) -> list[list[float]]:
    """Embed a list of texts using the same model as notebook 003."""
    response = w.serving_endpoints.query(name=EMBED_MODEL, input=texts)
    return [item.embedding for item in response.data]

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def retrieve(query: str, top_k: int = TOP_K) -> list[dict]:
    """Embed the query and return the top-k most similar failure records from Delta."""
    query_emb = np.array(embed([query])[0])
    emb_pd = spark.table(EMBEDDINGS_TABLE).toPandas()
    emb_pd["score"] = emb_pd["embedding"].apply(
        lambda e: cosine_similarity(query_emb, np.array(e))
    )
    return emb_pd.nlargest(top_k, "score").to_dict(orient="records")

record_count = spark.table(EMBEDDINGS_TABLE).count()
print(f"Retrieval functions ready. Index size: {record_count} records in {EMBEDDINGS_TABLE}")

In [ ]:
# COMMAND ----------

# DBTITLE 1,LLM helpers
# DBTITLE 1,LLM helpers
RAG_SYSTEM_PROMPT = """You are a Databricks job monitoring assistant. Your job is to help engineers
understand job failures, root causes, and recommended actions.

Answer the question using ONLY the information in the context below.
If the context does not contain enough information, say:
"I don't have enough data to answer this - try re-running the embedding pipeline
to ensure recent failures are indexed."""

def build_context(docs: list[dict]) -> str:
    return "\n\n---\n\n".join(d["content"] for d in docs)

from databricks.sdk.service.serving import ChatMessage, ChatMessageRole

def call_llm(question: str, context: str) -> str:
    """Send a RAG prompt to the configured Databricks Foundation Model API LLM."""
    response = w.serving_endpoints.query(
        name=LLM_MODEL,
        messages=[
            ChatMessage(role=ChatMessageRole.SYSTEM, content=RAG_SYSTEM_PROMPT),
            ChatMessage(role=ChatMessageRole.USER,   content=f"Context:\n{context}\n\nQuestion: {question}"),
        ],
        max_tokens=512,
        temperature=0.1,
    )
    return response.choices[0].message.content

print(f"LLM helpers ready. Using: {LLM_MODEL}")

In [ ]:
# COMMAND ----------

# DBTITLE 1,Run example queries
# DBTITLE 1,Run example queries
ask("Which jobs failed and what were the error types?")
ask("Why did the customer_etl_daily job fail?")
ask("What actions should be taken for the failed jobs?")
ask("Who should be notified about the failures?")
ask("Which jobs failed due to NullPointerException?")

ask_with_sources("What actions to be done for failed jobs?")